In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words) #dataset size

32033

In [4]:
chars = sorted(list(set(''.join(words)))) #build the vocabulary of characters
stoi = {s:i+1 for i,s in enumerate(chars)} #letter to integer(index) mapping    
stoi['.'] = 0 #index 0 will be reserved for the end of a word and start of the word
itos = {i:s for s,i in stoi.items()} # reverse mapping from integer(index) to letter
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [ ]:
block_size = 3 # context length: how many characters do we take to predict the next one? #upgrade to bigram level
X, Y = [], [] #inputs and targets split variables
for w in words[:5]:
  
  #print(w)
  context = [0] * block_size 
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [8]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [9]:
C = torch.randn((27, 2))

In [10]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [11]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [12]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [13]:
h

tensor([[ 0.9240, -0.6596, -0.4626,  ..., -0.1191,  0.9259,  0.5802],
        [-0.1852, -0.8311, -0.0764,  ..., -0.6527,  0.0912,  0.8482],
        [ 0.2664, -0.9156, -0.8328,  ..., -0.7643,  0.1809,  0.4692],
        ...,
        [-0.9535, -0.1291,  0.8330,  ..., -0.9749, -0.7530, -0.0990],
        [ 0.2898, -0.6150, -0.7950,  ..., -0.8586,  0.9042, -0.9594],
        [-0.9357, -0.9996, -0.9981,  ..., -0.4202, -0.6498,  0.5911]])

In [14]:
h.shape

torch.Size([32, 100])

In [15]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [16]:
logits = h @ W2 + b2

In [17]:
logits.shape

torch.Size([32, 27])

In [18]:
counts = logits.exp()

In [19]:
prob = counts / counts.sum(1, keepdims=True)

In [20]:
prob.shape

torch.Size([32, 27])

In [21]:
Y   #the actual target values

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [22]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(15.9514)

In [23]:

Xtr.shape, Ytr.shape # dataset

(torch.Size([182580, 3]), torch.Size([182580]))

In [24]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 10), generator=g) #embeddings matrix   
W1 = torch.randn((30, 200), generator=g) #hidden layer weights
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g) #second hidden layer weights
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2] 

In [25]:
sum(p.nelement() for p in parameters) # number of parameters in total

11897